# Module 04 Lab - Exploratory Data Analysis

**Objective:** To learn how to explore a dataset to find patterns, anomalies, and insights before modeling. EDA is like being a detective; you are looking for clues in the data that will help you build a better model.

**This lab is fully coded.** Your task is to run each cell, read the detailed explanations, understand the purpose of each visualization, and then complete the experimentation section.

## Part 1: Setup and Data Loading
**What is Exploratory Data Analysis (EDA)?**

EDA is the process of using summary statistics and visualizations to understand a dataset's main characteristics.

Before you can build a model, you need to understand your data.

What stories does it tell?

Are there errors or missing values?

Are there strong relationships between variables?

EDA helps answer these questions.

We will use the famous Titanic dataset for this lab. It contains information about passengers and, crucially, whether they survived the disaster.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
# Seaborn is a library built on top of Matplotlib that makes creating beautiful plots easier.

# Load the dataset directly from a URL
df = pd.read_csv('https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv')

print("--- First 5 Rows ---")
print(df.head())

print("--- Basic Info ---")
# .info() is a great first command. It tells us the column names, how many non-null values are in each column, and their data types.
# Notice that 'Age' and 'Cabin' have missing values!
df.info()

## Part 2: Descriptive Statistics

Let's start by getting a high-level numerical summary of the data. The `.describe()` method is perfect for this. It calculates statistics like mean, standard deviation, min, and max for the numerical columns.

In [ ]:
# Get summary statistics for numerical columns
print("--- Descriptive Statistics ---")
print(df.describe())

print("--- Key Insights from Statistics ---")
print(f"The average age of a passenger was {df['Age'].mean():.1f} years.")
print(f"The overall survival rate was {df['Survived'].mean():.1%}.")
print(f"Fares ranged from ${df['Fare'].min()} to a whopping ${df['Fare'].max()}.")

## Part 3: Visual EDA - Telling Stories with Plots

Numbers are great, but plots make patterns and relationships immediately obvious. The goal of visual EDA is to turn data into insights.

**A Note on Plotting Libraries:**
*   **Matplotlib:** The foundational library, gives you full control over everything.
*   **Seaborn:** Built on Matplotlib, it provides a simpler, high-level interface for creating common statistical plots. We will use Seaborn for its ease of use and attractive defaults.

### Visualization 1: How many survived?

A simple `countplot` is the best way to see the distribution of a categorical variable, like our target `Survived`.

In [ ]:
sns.set_style('whitegrid')  # Sets a nice visual style for our plots
plt.figure(figsize=(8, 6))
sns.countplot(x='Survived', data=df)
plt.title('Survival Distribution (0 = Died, 1 = Survived)')
plt.show()
print("Insight: Far more people died than survived. This is an example of an imbalanced dataset, which can sometimes be a challenge for machine learning models.")

### Visualization 2: Does passenger class matter for survival?

Now we want to see if there's a relationship between two variables. We can use the `hue` parameter in `countplot` to split the bars by another category.

In [ ]:
plt.figure(figsize=(10, 6))
sns.countplot(x='Pclass', hue='Survived', data=df)
plt.title('Survival by Passenger Class')
plt.legend(['Died', 'Survived'])
plt.show()
print("Insight: This is a very strong pattern. 1st class passengers had a much higher chance of survival compared to 3rd class passengers. Money seems to have made a difference.")

### Visualization 3: What about gender?

The 'women and children first' mantra is famous. Let's see if the data supports it.

In [ ]:
plt.figure(figsize=(10, 6))
sns.countplot(x='Sex', hue='Survived', data=df)
plt.title('Survival by Gender')
plt.legend(['Died', 'Survived'])
plt.show()
print("Insight: The pattern is undeniable. A much higher proportion of females survived compared to males. This is another very strong predictor.")

### Visualization 4: How does age play a role?

For a continuous variable like `Age`, a histogram is a great way to see its distribution.

In [ ]:
# A FacetGrid allows us to create multiple plots side-by-side to compare distributions.
# Here, we create one histogram for passengers who died (col='Survived'=0) and one for those who survived (col='Survived'=1).
g = sns.FacetGrid(df, col='Survived', height=6)
g.map(plt.hist, 'Age', bins=20)
plt.show()
print("Insight: The age distribution for those who did not survive is centered around the 20-40 age range. For those who survived, there is a noticeable spike for young children. This supports the 'children' part of the mantra.")

## Part 4: Student Experimentation

**Instructions:** Create your own visualizations to explore other relationships in the data.

### Experiment 1: Port of Embarkation

1.  The `Embarked` column tells you where a passenger boarded the ship (C = Cherbourg, Q = Queenstown, S = Southampton).
2.  Create a `countplot` to see how survival rates differed by the port of embarkation. Does where they boarded seem to be related to their survival?

In [ ]:
# --- Experiment 1: Survival by Port of Embarkation ---
plt.figure(figsize=(10, 6))
sns.countplot(x='Embarked', hue='Survived', data=df, order=['S', 'C', 'Q'])
plt.title('Survival by Port of Embarkation (S = Southampton, C = Cherbourg, Q = Queenstown)')
plt.legend(['Died', 'Survived'])
plt.show()

# Survival rate per port (counts alone can be misleading because port sizes differ)
print("Survival rate by port:")
print(df.groupby('Embarked')['Survived'].mean().round(3))

# Check whether class explains the difference
print("\nPassenger class mix by port (proportion):")
print(pd.crosstab(df['Embarked'], df['Pclass'], normalize='index').round(2))

print("\nInsight: Passengers who boarded at Cherbourg (C) had the highest survival rate (~55%), "
      "while Southampton (S) had the lowest (~34%). However, the port itself probably did not cause survival: "
      "about half of the Cherbourg passengers were 1st class, while most Southampton and Queenstown passengers were 3rd class. "
      "Embarkation is related to survival mostly through passenger class.")

### Experiment 2: Fare vs. Survival

1.  `Fare` is a continuous numerical variable.
2.  A `boxplot` or `violinplot` is a great way to see the distribution of fares for those who survived vs. those who didn't.
3.  Create one of these plots to compare the `Fare` distribution by `Survived`. What does it tell you?

In [ ]:
# --- Experiment 2: Fare distribution by Survival ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Boxplot with all fares
sns.boxplot(x='Survived', y='Fare', data=df, ax=axes[0])
axes[0].set_title('Fare by Survival (0 = Died, 1 = Survived)')

# Violinplot limited to fares <= 300 so the extreme outliers ($512) don't squash the plot
sns.violinplot(x='Survived', y='Fare', data=df[df['Fare'] <= 300], ax=axes[1])
axes[1].set_title('Fare by Survival (fares <= $300)')

plt.tight_layout()
plt.show()

print("Median fare by survival:")
print(df.groupby('Survived')['Fare'].median())

print("\nInsight: Survivors paid noticeably higher fares (median about $26 vs. about $10 for those who died). "
      "Most passengers who died paid very low fares, while survivors show a much wider spread with many high fares. "
      "There are also extreme outliers (up to ~$512). Since fare is closely tied to passenger class, "
      "this confirms that wealthier passengers had a better chance of surviving.")

## 📝 Knowledge Check

**Instructions:** Answer the following questions in this markdown cell.

1.  **What is the primary goal of Exploratory Data Analysis (EDA)?**
2.  **Based on the plots in this lab, what kind of person had the best chance of surviving the Titanic?** (Describe them in terms of class, gender, and age).
3.  **Why is it important to visualize data instead of just looking at summary statistics?** What can a plot show you that a number like 'mean' or 'count' can't?

---

**Answers:**

**1.** The primary goal of EDA is to understand the dataset before building a model. We use summary statistics and visualizations to learn the structure of the data, find patterns and relationships between variables, and spot problems such as missing values (e.g., `Age` and `Cabin`), outliers (e.g., very high fares), and imbalanced classes. These findings guide data cleaning, feature selection, and the choice of model.

**2.** The person with the best chance of surviving was a **female, traveling in 1st class, and a young child or adult woman** (not a 20–40 year-old man). From the plots: females survived at a much higher rate than males, 1st class passengers survived far more often than 3rd class passengers, and young children had a noticeable survival spike. Passengers who paid higher fares (and those who boarded at Cherbourg, who were mostly 1st class) also survived more often. The lowest chance belonged to adult men in 3rd class.

**3.** Summary statistics compress the data into a few numbers and can hide important details. A plot shows the full **shape** of the distribution: skewness, multiple peaks, clusters, gaps, and outliers. For example, the mean fare (~$32) hides that most passengers paid under $15 while a few paid over $500, and the mean age (~29.7) hides the spike of young children among survivors. Plots also make relationships between variables (like survival by class and gender) obvious at a glance, which helps us find patterns and errors that a single number like 'mean' or 'count' cannot reveal.